In [ ]:
5

In [ ]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse

visited = set()

def dfs_crawl(url, base_url, depth=0, max_depth=3):

    if depth > max_depth or url in visited:
        return

    print("Visiting:", url)

    visited.add(url)

    response = requests.get(url)

    soup = BeautifulSoup(response.text, "html.parser")

    links = soup.find_all("a", href=True)

    for link in links:

        absolute_url = urljoin(url, link["href"])

        if urlparse(absolute_url).netloc == urlparse(base_url).netloc:
            dfs_crawl(absolute_url, base_url, depth + 1, max_depth)

In [ ]:
dfs_crawl("https://www.tensorflow.org/tutorials",
          "https://www.tensorflow.org",
          max_depth=2)

Simple DFS - without filtering

In [11]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse

visited = set()

def dfs_crawl(url, base_url, depth=0, max_depth=3):

    if depth > max_depth or url in visited:
        return

    print(f"\n{'  '*depth}Visiting: {url}")

    visited.add(url)

    try:
        response = requests.get(url, timeout=5)

        soup = BeautifulSoup(response.text, "html.parser")

        # Extract text
        text = soup.get_text(separator=" ", strip=True)

        # Print sample of text
        sample = text[:300]
        print(f"{'  '*depth}Sample Text: {sample}\n")

        # Extract links
        links = soup.find_all("a", href=True)

        # Only first 3 links
        links = links[:3]

        for link in links:

            absolute_url = urljoin(url, link["href"])

            if urlparse(absolute_url).netloc == urlparse(base_url).netloc:
                dfs_crawl(absolute_url, base_url, depth + 1, max_depth)

    except Exception as e:
        print("Error:", e)


# Start crawling
dfs_crawl(
    "https://www.tensorflow.org/tutorials",
    "https://www.tensorflow.org",
    max_depth=2
)


Visiting: https://www.tensorflow.org/tutorials
Sample Text: Tutorials  |  TensorFlow Core Skip to main content Install Learn Introduction New to TensorFlow? Tutorials Learn how to use TensorFlow with end-to-end examples Guide Learn framework concepts and components Learn ML Educational resources to master your path with TensorFlow API TensorFlow (v2.16.1) Ve


  Visiting: https://www.tensorflow.org/tutorials#main-content
  Sample Text: Tutorials  |  TensorFlow Core Skip to main content Install Learn Introduction New to TensorFlow? Tutorials Learn how to use TensorFlow with end-to-end examples Guide Learn framework concepts and components Learn ML Educational resources to master your path with TensorFlow API TensorFlow (v2.16.1) Ve


    Visiting: https://www.tensorflow.org/
    Sample Text: TensorFlow Skip to main content Install Learn Introduction New to TensorFlow? Tutorials Learn how to use TensorFlow with end-to-end examples Guide Learn framework concepts and components Learn ML E

DFS paths with URL Filter (#-removed)

In [12]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse, urlunparse

visited = set()

def normalize_and_filter_url(base_url, current_url, href):

    # Convert to absolute URL
    absolute_url = urljoin(current_url, href)
    parsed = urlparse(absolute_url)

    # Remove fragment (#...)
    parsed = parsed._replace(fragment="")

    # Remove query params (?hl=en)
    parsed = parsed._replace(query="")

    # Normalize path (remove trailing slash)
    normalized_path = parsed.path.rstrip("/")
    parsed = parsed._replace(path=normalized_path)

    clean_url = urlunparse(parsed)

    # Same domain check
    if urlparse(clean_url).netloc != urlparse(base_url).netloc:
        return None

    # Restrict to tutorials path
    if not clean_url.startswith(base_url.rstrip("/")):
        return None

    # Skip non-html resources
    if clean_url.endswith((".pdf", ".jpg", ".png", ".zip")):
        return None

    return clean_url


def dfs_crawl(url, base_url, depth=0, max_depth=3):

    # Normalize current URL as well (IMPORTANT)
    url = url.rstrip("/")

    if depth > max_depth or url in visited:
        return

    print(f"\n{'  '*depth}Visiting: {url}")

    visited.add(url)

    try:
        headers = {
            "User-Agent": "Mozilla/5.0"
        }

        response = requests.get(url, headers=headers, timeout=5)

        soup = BeautifulSoup(response.text, "html.parser")

        # Remove unwanted tags
        for tag in soup(["script", "style", "nav", "footer"]):
            tag.decompose()

        # Extract text
        text = soup.get_text(separator=" ", strip=True)

        # Print sample text
        sample = text[:300]
        print(f"{'  '*depth}Sample Text: {sample}\n")

        # Extract links
        raw_links = soup.find_all("a", href=True)

        clean_links = []
        seen_links = set()

        for link in raw_links:

            clean_url = normalize_and_filter_url(base_url, url, link["href"])

            # Skip invalid
            if not clean_url:
                continue

            # Skip duplicates (VERY IMPORTANT)
            if clean_url in seen_links:
                continue

            # Skip already visited
            if clean_url in visited:
                continue

            # Add to lists
            seen_links.add(clean_url)
            clean_links.append(clean_url)

            # Limit AFTER filtering
            if len(clean_links) == 3:
                break

        # Debug: show children
        for cl in clean_links:
            print(f"{'  '*depth}→ Next: {cl}")

        # DFS recursion
        for link in clean_links:
            dfs_crawl(link, base_url, depth + 1, max_depth)

    except Exception as e:
        print("Error:", e)


# Start crawling
dfs_crawl(
    "https://www.tensorflow.org/tutorials",
    "https://www.tensorflow.org/tutorials",
    max_depth=2
)


Visiting: https://www.tensorflow.org/tutorials
Sample Text: Tutorials  |  TensorFlow Core Skip to main content / English Español – América Latina Français Indonesia Italiano Polski Português – Brasil Tiếng Việt Türkçe Русский עברית العربيّة فارسی हिंदी বাংলা ภาษาไทย 中文 – 简体 中文 – 繁體 日本語 한국어 GitHub Sign in TensorFlow Core An open source machine learning libr

→ Next: https://www.tensorflow.org/tutorials/quickstart/beginner
→ Next: https://www.tensorflow.org/tutorials/keras/classification
→ Next: https://www.tensorflow.org/tutorials/load_data/csv

  Visiting: https://www.tensorflow.org/tutorials/quickstart/beginner
  Sample Text: TensorFlow 2 quickstart for beginners  |  TensorFlow Core Skip to main content / English Español – América Latina Français Indonesia Italiano Polski Português – Brasil Tiếng Việt Türkçe Русский עברית العربيّة فارسی हिंदी বাংলা ภาษาไทย 中文 – 简体 日本語 한국어 GitHub Sign in TensorFlow Core TensorFlow Learn

  → Next: https://www.tensorflow.org/tutorials/keras
  → Nex

Header & Footer Removed

In [14]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse, urlunparse

visited = set()

def normalize_and_filter_url(base_url, current_url, href):

    # Convert to absolute URL
    absolute_url = urljoin(current_url, href)
    parsed = urlparse(absolute_url)

    # Remove fragment (#...)
    parsed = parsed._replace(fragment="")

    # Remove query params (?hl=en)
    parsed = parsed._replace(query="")

    # Normalize path (remove trailing slash)
    normalized_path = parsed.path.rstrip("/")
    parsed = parsed._replace(path=normalized_path)

    clean_url = urlunparse(parsed)

    # Same domain check
    if urlparse(clean_url).netloc != urlparse(base_url).netloc:
        return None

    # Restrict to tutorials path
    if not clean_url.startswith(base_url.rstrip("/")):
        return None

    # Skip non-html resources
    if clean_url.endswith((".pdf", ".jpg", ".png", ".zip")):
        return None

    return clean_url


def dfs_crawl(url, base_url, depth=0, max_depth=3):

    # Normalize current URL as well (IMPORTANT)
    url = url.rstrip("/")

    if depth > max_depth or url in visited:
        return

    print(f"\n{'  '*depth}Visiting: {url}")

    visited.add(url)

    try:
        headers = {
            "User-Agent": "Mozilla/5.0"
        }

        response = requests.get(url, headers=headers, timeout=5)

        soup = BeautifulSoup(response.text, "html.parser")

        # Remove unwanted tags (stronger cleaning)
        for tag in soup(["script", "style", "nav", "footer", "header", "aside"]):
            tag.decompose()

        # Try extracting MAIN content only
        main = soup.find("main")

        if main:
            text = main.get_text(" ", strip=True)
        else:
            text = soup.get_text(" ", strip=True)

        # Clean extra whitespace
        text = " ".join(text.split())

        # Print sample text
        sample = text[:300]
        print(f"{'  '*depth}Sample Text: {sample}\n")

        # Extract links
        raw_links = soup.find_all("a", href=True)

        clean_links = []
        seen_links = set()

        for link in raw_links:

            clean_url = normalize_and_filter_url(base_url, url, link["href"])

            # Skip invalid
            if not clean_url:
                continue

            # Skip duplicates (VERY IMPORTANT)
            if clean_url in seen_links:
                continue

            # Skip already visited
            if clean_url in visited:
                continue

            # Add to lists
            seen_links.add(clean_url)
            clean_links.append(clean_url)

            # Limit AFTER filtering
            if len(clean_links) == 3:
                break

        # Debug: show children
        for cl in clean_links:
            print(f"{'  '*depth}→ Next: {cl}")

        # DFS recursion
        for link in clean_links:
            dfs_crawl(link, base_url, depth + 1, max_depth)

    except Exception as e:
        print("Error:", e)


# Start crawling
dfs_crawl(
    "https://www.tensorflow.org/tutorials",
    "https://www.tensorflow.org/tutorials",
    max_depth=2
)


Visiting: https://www.tensorflow.org/tutorials
Sample Text: TensorFlow Learn TensorFlow Core Stay organized with collections Save and categorize content based on your preferences. The TensorFlow tutorials are written as Jupyter notebooks and run directly in Google Colab—a hosted notebook environment that requires no setup. At the top of each tutorial, you'll

→ Next: https://www.tensorflow.org/tutorials/quickstart/beginner
→ Next: https://www.tensorflow.org/tutorials/keras/classification
→ Next: https://www.tensorflow.org/tutorials/load_data/csv

  Visiting: https://www.tensorflow.org/tutorials/quickstart/beginner
  Sample Text: TensorFlow Learn TensorFlow Core TensorFlow 2 quickstart for beginners Stay organized with collections Save and categorize content based on your preferences. View on TensorFlow.org Run in Google Colab View source on GitHub Download notebook This short introduction uses Keras to: Load a prebuilt data

  → Next: https://www.tensorflow.org/tutorials/keras
  → Nex

URL FILTERING - FilterChain, DomainFilter, URLPatternFilter, ContentTypeFilter

In [1]:
5

5

In [2]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse, urlunparse
import importlib
import filters
importlib.reload(filters)
from filters import FilterChain, DomainFilter, URLPatternFilter, ContentTypeFilter

visited = set()

filter_chain = FilterChain([
    DomainFilter("https://www.tensorflow.org/tutorials"),
    URLPatternFilter("https://www.tensorflow.org/tutorials"),
    ContentTypeFilter()
],
debug=True #to check the link is passing through the filters or not
)

def normalize_and_filter_url(base_url, current_url, href):

    absolute_url = urljoin(current_url, href)
    parsed = urlparse(absolute_url)

    parsed = parsed._replace(fragment="")
    parsed = parsed._replace(query="")

    normalized_path = parsed.path.rstrip("/")
    parsed = parsed._replace(path=normalized_path)

    clean_url = urlunparse(parsed)

    # 🔥 Apply filter chain
    if not filter_chain.allow(clean_url):
        return None

    return clean_url


def dfs_crawl(url, base_url, depth=0, max_depth=3):

    # Normalize current URL as well (IMPORTANT)
    url = url.rstrip("/")

    if depth > max_depth or url in visited:
        return

    print(f"\n{'  '*depth}Visiting: {url}")

    visited.add(url)

    try:
        headers = {
            "User-Agent": "Mozilla/5.0"
        }

        response = requests.get(url, headers=headers, timeout=5)

        soup = BeautifulSoup(response.text, "html.parser")

        # Remove unwanted tags (stronger cleaning)
        for tag in soup(["script", "style", "nav", "footer", "header", "aside"]):
            tag.decompose()

        # Try extracting MAIN content only
        main = soup.find("main")

        if main:
            text = main.get_text(" ", strip=True)
        else:
            text = soup.get_text(" ", strip=True)

        # Clean extra whitespace
        text = " ".join(text.split())

        # Print sample text
        sample = text[:300]
        print(f"{'  '*depth}Sample Text: {sample}\n")

        # Extract links
        raw_links = soup.find_all("a", href=True)

        clean_links = []
        seen_links = set()

        for link in raw_links:

            clean_url = normalize_and_filter_url(base_url, url, link["href"])

            # Skip invalid
            if not clean_url:
                continue

            # Skip duplicates (VERY IMPORTANT)
            if clean_url in seen_links:
                continue

            # Skip already visited
            if clean_url in visited:
                continue

            # Add to lists
            seen_links.add(clean_url)
            clean_links.append(clean_url)

            # Limit AFTER filtering
            if len(clean_links) == 3:
                break

        # Debug: show children
        for cl in clean_links:
            print(f"{'  '*depth}→ Next: {cl}")

        # DFS recursion
        for link in clean_links:
            dfs_crawl(link, base_url, depth + 1, max_depth)

    except Exception as e:
        print("Error:", e)


# Start crawling
dfs_crawl(
    "https://www.tensorflow.org/tutorials",
    "https://www.tensorflow.org/tutorials",
    max_depth=2
)

d:\Adrta\rag_env\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.0.1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(



Visiting: https://www.tensorflow.org/tutorials
Sample Text: TensorFlow Learn TensorFlow Core Stay organized with collections Save and categorize content based on your preferences. The TensorFlow tutorials are written as Jupyter notebooks and run directly in Google Colab—a hosted notebook environment that requires no setup. At the top of each tutorial, you'll

[Filter: DomainFilter] https://www.tensorflow.org/tutorials → True
[Filter: URLPatternFilter] https://www.tensorflow.org/tutorials → True
[Filter: ContentTypeFilter] https://www.tensorflow.org/tutorials → True
[Filter: DomainFilter] https://www.tensorflow.org → True
[Filter: URLPatternFilter] https://www.tensorflow.org → False
[Filter: DomainFilter] https://github.com/tensorflow → False
[Filter: DomainFilter] https://www.tensorflow.org/tutorials → True
[Filter: URLPatternFilter] https://www.tensorflow.org/tutorials → True
[Filter: ContentTypeFilter] https://www.tensorflow.org/tutorials → True
[Filter: DomainFilter] https://www.te